# 03 — Compare phenology peaks with a maximum value composite

Compare the smoothed seasonal NDVI peak with a maximum value composite (MVC) calculated from cloud-masked Landsat observations around the AOI peak date.


## Inputs, outputs, and execution order

**Inputs:** The boundary, per-year phenology CSVs and vPOS rasters from 02, and live Landsat data.

**Outputs:** MVC GeoTIFFs, POS–MVC comparison PNGs, yearly comparison CSVs, and a combined summary under `pheno/graph/`.

Run cells from top to bottom in a fresh kernel. Open this notebook beside `workflow_config.json` and `workflow_support.py`. The configuration uses Mion study settings; the original boundary and final criteria table were not supplied. See the README for the small example and for adapting the workflow.

### How to interpret this step

This is a **peak-window MVC**, not a calendar-year maximum. The first successful load uses ±1 month around the AOI peak; load failures trigger ±2 and ±3 month attempts. NDVI is clipped to [0, 1] as in the original. POS_NDVI comes from the AOI spatial-median curve, while MVC_NDVI is the spatial mean of pixelwise maxima. Differences reflect processing and summary choices, so they are not an accuracy assessment. MVC_DOY is the integer-truncated mean acquisition day of the pixelwise maxima.


## 1. Import packages

Use the Digital Earth Africa Sandbox for steps that access the data cube. Local raster processing uses the packages listed in `environment.yml`.


In [ ]:
%matplotlib inline
import os
import gc
import psutil
import numpy as np
import pandas as pd
import rioxarray as rxr
import matplotlib.pyplot as plt
import datacube
from deafrica_tools.datahandling import load_ard
from deafrica_tools.bandindices import calculate_indices
from deafrica_tools.spatial import xr_rasterize


## 2. Load shared settings

The settings below describe the run. Edit the JSON configuration before continuing; later parameter cells explain the settings used by this notebook.


In [ ]:
# Edit workflow_config.json once, then restart and rerun each notebook.
from workflow_support import load_config, require_files, load_aoi, assert_same_grid
cfg = load_config()
print(f"Study: {cfg['study_id']} | Years: {cfg['start_year']}-{cfg['end_year']}")
print(f"Output directory: {cfg['output_dir']}")


## 3. Connect and define the study boundary

Use the same boundary as notebook 01 so the comparison covers the same area.


In [ ]:
dc = datacube.Datacube(app='mvc')

# The helper merges all polygon features and converts their CRS to EPSG:4326.
# Supply your own GeoJSON, GeoPackage, or complete shapefile in the configuration.
geopolygon_gdf = load_aoi(cfg["aoi_vector"])
minx, miny, maxx, maxy = geopolygon_gdf.total_bounds
lat_range = (miny, maxy)
lon_range = (minx, maxx)
print("Query bounds (longitude, latitude):", lon_range, lat_range)


## 4. Locate phenology inputs

All requested phenology inputs are checked before loading Landsat data. Notebook 02 now uses the same configured year range.


In [ ]:
veg_proxy = "NDVI"
area_of_interest = cfg["study_id"]
output_folder = f"{cfg['output_dir']}/pheno/graph"
for year in range(cfg["start_year"], cfg["end_year"] + 1):
    require_files(f"{output_folder}/{area_of_interest}_csv_{veg_proxy}_{year}.csv",
                  f"{output_folder}/{area_of_interest}_{veg_proxy}_POS_{year}.tif")


## 5. Define the comparison

Each year is processed separately. The output table records the successful search window and number of loaded observations. Empty or invalid observations can still prevent a valid comparison; inspect data coverage.


In [ ]:
# =========================================================
# HELPER: RAM MONITOR
# =========================================================
def print_ram(stage=""):
    process = psutil.Process(os.getpid())
    mem_gb = process.memory_info().rss / (1024 ** 3)
    print(f"[RAM] {stage}: {mem_gb:.2f} GB")


# =========================================================
# HELPER: PROCESS ONE YEAR
# =========================================================
def process_year(year, dc, lon_range, lat_range, geopolygon_gdf):
    print("\n" + "=" * 70)
    print(f"Processing year: {year}")
    print("=" * 70)
    print_ram("Start of year")
    
    phenology_csv_path = (
        f"{output_folder}/{area_of_interest}_csv_{veg_proxy}_{year}.csv"
    )
    pos_raster_path = (
        f"{output_folder}/{area_of_interest}_{veg_proxy}_POS_{year}.tif"
    )
    mvc_raster_path = (
        f"{output_folder}/{area_of_interest}_{veg_proxy}_MVC_{year}.tif"
    )
    png_out = (
        f"{output_folder}/{area_of_interest}_{veg_proxy}_POS_MVC_{year}.png"
    )
    summary_out = (
        f"{output_folder}/{area_of_interest}_{veg_proxy}_pos_mvc_summary_{year}.csv"
    )

    ds = None
    mask = None
    ndvi = None
    mvc = None
    pos_ds = None
    pos_img = None
    fig = None
    axes = None

    try:
        # -----------------------------
        # Load POS metadata
        # -----------------------------
        phenology_csv = pd.read_csv(phenology_csv_path)
        pos_date_scalar = pd.to_datetime(phenology_csv["POS"].iloc[0])
        pos_value = float(phenology_csv["vPOS"].iloc[0])
        pos_doy_value = int(pos_date_scalar.dayofyear)

        # -----------------------------
        # Load Landsat ARD with retries
        # -----------------------------
        ds = None
        for month_buffer in cfg["mvc_month_buffers"]:
            try:
                start = pos_date_scalar - pd.DateOffset(months=month_buffer)
                end = pos_date_scalar + pd.DateOffset(months=month_buffer)
                time_period = (start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"))
                print("Time period:", time_period)

                ds = load_ard(
                    dc=dc,
                    products=cfg["landsat_products"],
                    measurements=["red", "nir"],  # faster than loading RGB too
                    mask_filters=[("opening", 3), ("dilation", 3)],
                    time=time_period,
                    x=lon_range,
                    y=lat_range,
                    output_crs=cfg["output_crs"],
                    resolution=tuple(cfg["resolution"]),
                    dask_chunks=cfg["dask_chunks"],
                )
                break

            except Exception as e:
                print(f"Load attempt failed for year {year} with ±{month_buffer} month(s): {e}")

        if ds is None:
            raise ValueError(f"Could not load data for year {year}")

        print_ram("After load_ard")

        # -----------------------------
        # AOI mask
        # -----------------------------
        mask = xr_rasterize(geopolygon_gdf, ds)
        ds = ds.where(mask)
        print_ram("After masking")

        # Mask no longer needed
        mask = None
        gc.collect()

        # -----------------------------
        # NDVI
        # -----------------------------
        ndvi = calculate_indices(ds, index=veg_proxy, satellite_mission="ls").NDVI
        ndvi = ndvi.clip(min=0, max=1)
        print_ram("After NDVI")

        # Raw dataset no longer needed
        ds = None
        gc.collect()
        print_ram("After releasing ds")

        # -----------------------------
        # MVC raster
        # -----------------------------
        mvc = ndvi.max(dim="time", skipna=True)
        mvc.rio.write_nodata(np.nan).rio.to_raster(mvc_raster_path)
        print(f"Saved: {mvc_raster_path}")
        print_ram("After MVC save")

        # -----------------------------
        # Scalar summaries
        # -----------------------------
        mvc_value = float(mvc.mean(dim=("y", "x"), skipna=True).values)

        valid_any = ndvi.notnull().any("time")
        filled = ndvi.where(valid_any).fillna(-1e9)
        mvc_index = filled.argmax("time")
        mvc_time_map = ndvi["time"].isel(time=mvc_index)

        mvc_doy_value = int(
            mvc_time_map.dt.dayofyear.where(valid_any).mean(skipna=True).values
        )

        ndvi_diff = float(mvc_value - pos_value)
        doy_diff = int(mvc_doy_value - pos_doy_value)

        # Record the successful observation window; widening changes the MVC comparison.
        df = pd.DataFrame({
            "Window_start": [start.strftime("%Y-%m-%d")],
            "Window_end": [end.strftime("%Y-%m-%d")],
            "Month_buffer": [month_buffer],
            "Observation_count": [ndvi.sizes["time"]],
            "Year": [year],
            "POS_NDVI": [pos_value],
            "MVC_NDVI": [mvc_value],
            "Difference_NDVI": [ndvi_diff],
            "POS_DOY": [pos_doy_value],
            "MVC_DOY": [mvc_doy_value],
            "Difference_DOY": [doy_diff],
        })

        print(df)
        df.to_csv(summary_out, index=False)
        print(f"Saved: {summary_out}")

        # -----------------------------
        # Release heavy NDVI intermediates before plotting
        # -----------------------------
        valid_any = None
        filled = None
        mvc_index = None
        mvc_time_map = None
        gc.collect()
        print_ram("After scalar summary cleanup")

        # -----------------------------
        # Plot POS vs MVC
        # Avoid reopening MVC raster; use mvc already in memory
        # -----------------------------
        pos_ds = rxr.open_rasterio(pos_raster_path, masked=True)
        pos_img = pos_ds.squeeze().load()
        try:
            pos_ds.close()
        except Exception:
            pass
        pos_ds = None

        assert_same_grid([pos_raster_path, mvc_raster_path])
        mvc_img = mvc.load()

        fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharey=True)

        im1 = axes[0].imshow(pos_img, cmap="RdYlGn", vmin=0, vmax=1)
        axes[0].set_title(
            f"{area_of_interest.upper()} – Peak of Season (POS), {year}",
            fontsize=15,
            fontweight="bold",
        )
        axes[0].set_xlabel("Column index")
        axes[0].set_ylabel("Row index")
        axes[0].grid(True, color="white", linewidth=0.3, alpha=0.6)
        plt.colorbar(im1, ax=axes[0], shrink=0.7)

        im2 = axes[1].imshow(mvc_img, cmap="RdYlGn", vmin=0, vmax=1)
        axes[1].set_title(
            f"{area_of_interest.upper()} – Maximum Value Composite (MVC), {year}",
            fontsize=15,
            fontweight="bold",
        )
        axes[1].set_xlabel("Column index")
        axes[1].grid(True, color="white", linewidth=0.3, alpha=0.6)
        plt.colorbar(im2, ax=axes[1], shrink=0.7)

        plt.tight_layout()
        plt.savefig(png_out, dpi=300, bbox_inches="tight")
        print(f"Saved: {png_out}")
        plt.close(fig)

        # Release plot arrays immediately
        pos_img = None
        mvc_img = None
        fig = None
        axes = None
        gc.collect()
        print_ram("After plot cleanup")

        # Release remaining arrays
        ndvi = None
        mvc = None
        df = None
        phenology_csv = None
        gc.collect()
        print_ram("End of year cleanup")

        print(f"✓ Finished processing year {year}")

    except Exception as e:
        print(f"Error processing year {year}: {e}")
        raise

    finally:
        plt.close("all")

        try:
            if pos_ds is not None:
                pos_ds.close()
        except Exception:
            pass

        ds = None
        mask = None
        ndvi = None
        mvc = None
        pos_ds = None
        pos_img = None
        fig = None
        axes = None

        gc.collect()
        print_ram("After final cleanup")
        print("✓ Memory cleared for next loop")


## 6. Run the requested years

Processing stops if a year fails, rather than reporting an apparently complete comparison.


In [ ]:
# =========================================================
# MAIN LOOP
# =========================================================
for year in range(cfg["start_year"], cfg["end_year"] + 1):
    process_year(
        year=year,
        dc=dc,
        lon_range=lon_range,
        lat_range=lat_range,
        geopolygon_gdf=geopolygon_gdf,
    )

print("\n✓ All yearly processing completed.")


## 7. Merge comparison tables

Rebuild the combined file using one table per configured year; never include the previous combined file in the inputs.


In [ ]:
csv_files = [f"{output_folder}/{area_of_interest}_{veg_proxy}_pos_mvc_summary_{y}.csv"
             for y in range(cfg["start_year"], cfg["end_year"] + 1)]
require_files(*csv_files)
df_all = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)
out_file = os.path.join(output_folder, "pos_mvc_summary_all_years.csv")
df_all.to_csv(out_file, index=False)
print(f"Merged {len(csv_files)} annual tables: {out_file}")


## Acknowledgements

We gratefully acknowledge the [Group on Earth Observations Land Degradation Neutrality (GEO-LDN) initiative](https://geo-ldn.org/) for its support of this research and the development of this land productivity monitoring workflow.

We acknowledge **Digital Earth Africa (DE Africa)** for providing the open satellite data, analytical tools, and Sandbox platform used in this work.

We also acknowledge the [University of Energy and Natural Resources (UENR), Ghana](https://uenr.edu.gh/), for its support of this research.


## Check the outputs before continuing

- Review warnings and errors, and confirm that every requested year or reporting period was written.
- Open an output raster and inspect its CRS, extent, nodata mask, and value range.
- Keep the configuration and software versions with your results. Run `python scripts/capture_environment.py` from the repository root.
- If you change the AOI, dates, metric, or thresholds, use a fresh output directory to avoid reusing older results.

See [README.md](README.md), [data requirements](docs/DATA_REQUIREMENTS.md), and [method details](docs/METHODS.md).

The supplied notebooks identify their code as Apache License 2.0; retain the original Digital Earth Africa attribution when reusing it. Dataset licences and acknowledgements apply separately. Use the Issues tab of the repository hosting this workflow for workflow-specific questions.

**Validation status:** documentation, syntax, and synthetic-data checks are described in `docs/VALIDATION.md`. This revised workflow has not been executed against the live Digital Earth Africa data cube. A displayed current date or package version is not evidence of a successful full run.
